# Trial Builder
Create and save `TrialDefinition` objects without running them.

Workflow:
1. Connect to CARLA and pick a map
2. Launch the interactive point picker to select route start/end
3. Configure trial parameters (name, weather, traffic)
4. Save the trial — `trial_demo.ipynb` loads and runs it

## 0 — CARLA Launch

In [1]:
import subprocess
subprocess.Popen("CarlaUE4.exe", shell=True)

<Popen: returncode: None args: 'CarlaUE4.exe'>

## 1 — Imports

In [2]:
import json
import re
import sys
from pathlib import Path

import carla

from MIREIA.config import Config
from MIREIA.simulation.trials import TrialDefinition

## 1 — CARLA Connection
Start CARLA before running this cell. Change `MAP_NAME` to load a different map.

In [3]:
MAP_NAME = "Town03"  # change to any available map

client = carla.Client(Config.CARLA_HOST, Config.CARLA_PORT)
client.set_timeout(15.0)

current_map = client.get_world().get_map().name.split("/")[-1]
if current_map != MAP_NAME:
    print(f"Loading {MAP_NAME} (current: {current_map})...")
    client.load_world(MAP_NAME)

world = client.get_world()
print(f"Connected. Map: {world.get_map().name.split('/')[-1]}")
print("Available maps:", [m.split("/")[-1] for m in client.get_available_maps()])

Loading Town03 (current: Town10HD_Opt)...
Connected. Map: Town03
Available maps: ['Town01', 'Town01_Opt', 'Town02', 'Town02_Opt', 'Town03', 'Town03_Opt', 'Town04', 'Town04_Opt', 'Town05', 'Town05_Opt', 'Town06', 'Town06_Opt', 'Town07', 'Town07_Opt', 'Town10HD', 'Town10HD_Opt', 'Town11', 'Town12', 'Town13', 'Town15', 'AnnotationColorLandscape']


## 2 — Existing Trials
Lists already-saved trial definitions.

In [4]:
trials_root = Path(Config.PATH_TO_TRIALS)
existing = sorted(p for p in trials_root.glob("*/trial.json") if p.is_file())

if not existing:
    print("No saved trials found.")
else:
    print(f"Found {len(existing)} trial(s):\n")
    for p in existing:
        data = json.loads(p.read_text(encoding="utf-8"))
        start = tuple(data.get("route_start", []))
        end   = tuple(data.get("route_end",   []))
        print(
            f"  {data['name']!r:40s}  "
            f"map={data.get('map_name','?')}  "
            f"weather={data.get('weather','?')}  "
            f"nv={data.get('n_vehicles','?')}  "
            f"start={start}  end={end}"
        )

Found 1 trial(s):

  'demo_trial_runner_streaming'             map=Town03  weather=ClearNoon  nv=30  start=(7.409, -64.4, 0.0)  end=(44.003, 50.549, 0.0)


## 3 — Pick Route Points
Launches a separate window showing the map waypoints.
**Left-click** to snap to the nearest waypoint. First click = start (green), second = end (red).
**Right-click** to undo. Close the window or press **Enter** to confirm.

The picked coordinates are parsed and stored in `route_start` and `route_end` automatically.

In [6]:
import os

routes_script = Path("MIREIA/simulation/routes.py")

# Override the matplotlib backend so the subprocess opens a real interactive
# window instead of inheriting Jupyter's inline backend (which closes instantly).
env = os.environ.copy()
env["MPLBACKEND"] = "TkAgg"

result = subprocess.run(
    [sys.executable, str(routes_script), "--host", Config.CARLA_HOST, "--port", str(Config.CARLA_PORT)],
    capture_output=True,
    text=True,
    env=env,
)

output = result.stdout + result.stderr
print(output.strip())

def _parse_carla_location(text: str, label: str) -> tuple[float, float, float] | None:
    pattern = rf"{label}\s*=\s*carla\.Location\(x=([\-\d.]+),\s*y=([\-\d.]+),\s*z=([\-\d.]+)\)"
    m = re.search(pattern, text)
    if m:
        return float(m.group(1)), float(m.group(2)), float(m.group(3))
    return None

route_start = _parse_carla_location(output, "START_CARLA")
route_end   = _parse_carla_location(output, "END_CARLA")

if route_start and route_end:
    print(f"\nParsed start: {route_start}")
    print(f"Parsed end:   {route_end}")
else:
    print("\n[warn] Could not parse coordinates — copy them from the output above and set route_start/route_end manually.")
    route_start = (0.0, 0.0, 0.0)
    route_end   = (0.0, 0.0, 0.0)

INFO:  Found the required file in cache!  Carla/Maps/Nav/Town03.bin 
Loaded 7092 waypoints from map: Carla/Maps/Town03
Click two points: START then END.
CLICK_1_WAYPOINT_ID = 309013290190582757
CLICK_1_MAP = (-111.698, -0.308)
CLICK_1_CARLA = carla.Location(x=-111.698, y=0.308, z=0.000)
CLICK_2_WAYPOINT_ID = 14283937585407435285
CLICK_2_MAP = (136.583, 72.299)
CLICK_2_CARLA = carla.Location(x=136.583, y=-72.299, z=0.000)
START_WAYPOINT_ID = 309013290190582757
END_WAYPOINT_ID = 14283937585407435285
START_CARLA = carla.Location(x=-111.698, y=0.308, z=0.000)
END_CARLA = carla.Location(x=136.583, y=-72.299, z=0.000)

Parsed start: (-111.698, 0.308, 0.0)
Parsed end:   (136.583, -72.299, 0.0)


### Manual Override
If the picker didn't run or you want to set coordinates by hand, edit and run this cell.

In [ ]:
# route_start = (7.409, -64.400, 0.0)
# route_end   = (44.003, 50.549, 0.0)
print(f"route_start = {route_start}")
print(f"route_end   = {route_end}")

## 4 — Trial Parameters
Edit these to define the trial. `name` must be unique — it becomes the folder name under `PATH_TO_TRIALS`.

Weather presets: `ClearNoon`, `CloudyNoon`, `WetNoon`, `HardRainNoon`, `CloudySunset`, `SoftRainSunset`, `ClearNight`, `CloudyNight`, `HardRainNight`.

In [7]:
TRIAL_NAME   = "my_trial_town03_clear"  # unique folder name
DESCRIPTION  = "Town03, clear noon, moderate traffic."
WEATHER      = "ClearNoon"
N_VEHICLES   = 30
N_PEDESTRIANS = 20
SEED         = 42

# Advanced — leave as defaults unless you need specific traffic behaviour
PCT_RUNNING   = 0.0   # fraction of pedestrians that run
PCT_CROSSING  = 0.0   # fraction of pedestrians that ignore traffic lights
SAFE_VEHICLES = True  # avoid collisions during vehicle spawning

## 5 — Create and Save Trial

In [8]:
trial = TrialDefinition(
    name=TRIAL_NAME,
    route_start=route_start,
    route_end=route_end,
    description=DESCRIPTION,
    map_name=MAP_NAME,
    weather=WEATHER,
    n_vehicles=N_VEHICLES,
    n_pedestrians=N_PEDESTRIANS,
    pct_running=PCT_RUNNING,
    pct_crossing=PCT_CROSSING,
    safe_vehicles=SAFE_VEHICLES,
    seed=SEED,
    sync_mode=True,
    fixed_delta=Config.SIM_FIXED_DELTA_SECONDS,
)

trial.save()
print(f"Saved: {trial.json_path}")
print(json.dumps(trial.to_dict(), indent=2))

Saved: t:\TFG\MIREIA\trials\my_trial_town03_clear\trial.json
{
  "name": "my_trial_town03_clear",
  "route_start": [
    -111.698,
    0.308,
    0.0
  ],
  "route_end": [
    136.583,
    -72.299,
    0.0
  ],
  "description": "Town03, clear noon, moderate traffic.",
  "map_name": "Town03",
  "weather": "ClearNoon",
  "n_vehicles": 30,
  "n_pedestrians": 20,
  "pct_running": 0.0,
  "pct_crossing": 0.0,
  "safe_vehicles": true,
  "seed": 42,
  "sync_mode": true,
  "fixed_delta": 0.05
}


## 6 — Review / Load an Existing Trial
Load any saved trial by name to inspect or copy its parameters.

In [9]:
load_name = TRIAL_NAME  # or replace with any other trial name

loaded = TrialDefinition.load(load_name)
print(f"Loaded: {loaded.name}")
print(json.dumps(loaded.to_dict(), indent=2))

Loaded: my_trial_town03_clear
{
  "name": "my_trial_town03_clear",
  "route_start": [
    -111.698,
    0.308,
    0.0
  ],
  "route_end": [
    136.583,
    -72.299,
    0.0
  ],
  "description": "Town03, clear noon, moderate traffic.",
  "map_name": "Town03",
  "weather": "ClearNoon",
  "n_vehicles": 30,
  "n_pedestrians": 20,
  "pct_running": 0.0,
  "pct_crossing": 0.0,
  "safe_vehicles": true,
  "seed": 42,
  "sync_mode": true,
  "fixed_delta": 0.05
}
